# NDVI lab: from reflectance to an index
Calculate the Normalised Difference Vegetation Index, inspect the effect of band values and preserve invalid pixels.

The arrays are synthetic reflectance values for learning. A real workflow must identify the sensor, band mapping, scale factors, product level and quality mask.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/spectral-indices'))
from spectral_indices import ndvi


## 1. Calculate the index

NDVI = (NIR − Red) / (NIR + Red). Larger values are often associated with green vegetation, but the index is not a direct measurement of biomass or plant health.

In [ ]:
nir = np.array([[0.62, 0.58, 0.42, 0.30], [0.60, 0.52, 0.35, 0.18], [0.55, 0.46, 0.25, 0.12], [0.48, 0.33, 0.20, 0.08]])
red = np.array([[0.10, 0.12, 0.18, 0.22], [0.11, 0.14, 0.20, 0.24], [0.13, 0.16, 0.21, 0.20], [0.15, 0.18, 0.16, 0.10]])
index = ndvi(nir, red)
fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
for ax, image, title in zip(axes, [nir, red, index], ['NIR reflectance', 'Red reflectance', 'NDVI']):
    im = ax.imshow(image, vmin=0 if title != 'NDVI' else -1, vmax=1)
    ax.set_title(title); ax.set_xticks(range(4)); ax.set_yticks(range(4)); fig.colorbar(im, ax=ax)
plt.show()


## 2. Preserve invalid pixels

A zero denominator, a non-finite input, a NoData sentinel or a false quality mask should not turn into a valid-looking index.

In [ ]:
nir_with_nodata = nir.copy(); red_with_nodata = red.copy()
nir_with_nodata[0, 0] = -9999
nir_with_nodata[1, 1] = np.nan
red_with_nodata[2, 2] = -nir_with_nodata[2, 2]  # denominator becomes zero
valid = np.ones(nir.shape, dtype=bool); valid[3, 3] = False
masked_index = ndvi(nir_with_nodata, red_with_nodata, nodata=-9999, valid_mask=valid)
print(masked_index)
print('Invalid cells:', np.argwhere(~np.isfinite(masked_index)).tolist())


## 3. Investigate

1. Why does the zero-denominator pixel remain undefined rather than becoming zero?
2. What does a high NDVI value suggest, and what does it not prove?
3. Which preprocessing steps must happen before the formula is meaningful?
4. Why should values outside the familiar range be investigated instead of silently clipped?
5. Which sensor, band IDs, scale factors and quality flags would you document in a real workflow?